# Flatten orders
Project nested book deltas into the canonical `Order` table with Arrow kernels.

In [ ]:
source = "market.books"
target = "market.orders"
start = None
end = None
catalog = "rekep"
catalog_properties = {}
merge_by = True
commit_row_size = 250_000

In [ ]:
import datetime

from pyiceberg.expressions import And, GreaterThanOrEqual, LessThan

from rekep.iceberg import IcebergDataset
from rekep.market import Book, Order


def _unix_ns(value, *, upper=False):
    if value is None:
        return None
    text = str(value)
    date_only = len(text) == 10
    instant = datetime.datetime.fromisoformat(text.replace("Z", "+00:00"))
    if instant.tzinfo is None:
        instant = instant.replace(tzinfo=datetime.UTC)
    instant = instant.astimezone(datetime.UTC)
    if upper and date_only:
        instant += datetime.timedelta(days=1)
    epoch = datetime.datetime(1970, 1, 1, tzinfo=datetime.UTC)
    return (instant - epoch) // datetime.timedelta(microseconds=1) * 1_000


def _filter(column="unix"):
    lower, upper = _unix_ns(start), _unix_ns(end, upper=True)
    predicates = []
    if lower is not None:
        predicates.append(GreaterThanOrEqual(column, lower))
    if upper is not None:
        predicates.append(LessThan(column, upper))
    return None if not predicates else predicates[0] if len(predicates) == 1 else And(*predicates)


books = IcebergDataset(name=source, catalog=catalog, properties=dict(catalog_properties))
orders = IcebergDataset(
    name=target,
    catalog=catalog,
    properties=dict(catalog_properties),
    struct=Order.into_field(),
    commit_row_size=commit_row_size,
    sort_by=("unix", "seq", "hash"),
)
counts = {"read": 0}


def _batches():
    reader = books.read_arrow_reader(
        Book.into_field(), row_filter=_filter(), order_by=("unix", "seq", "hash")
    )
    for batch in reader:
        flattened = Order.from_books_arrow_batch(batch)
        counts["read"] += flattened.num_rows
        if flattened.num_rows:
            yield flattened


written = orders.append_arrow_reader(
    _batches(), Order.into_field(), merge_by=merge_by, commit_row_size=commit_row_size
)
result = {"read": counts["read"], "written": written, "target": target}
result